# 🔐 Notebook 1: Storing Passwords — Bad, Better, Best

**The big question:** *"Where do I put the user's password?"*

Whenever someone signs up to your app, you need to remember **something** so that next time they log in you can confirm it is really them. The most obvious idea is to just save the password somewhere — but that turns out to be a terrible idea. In this notebook we walk through three approaches:

1. 🟥 **BAD** — store the password in plain text.
2. 🟨 **BETTER** — store a SHA-256 hash of the password.
3. 🟩 **BEST** — store a **bcrypt** hash with a per-user salt and a tunable cost factor.

## Learning objectives
- Understand why hashing is necessary (database leaks happen).
- See why a *fast* hash like SHA-256 is still bad for passwords.
- Use `bcrypt` correctly and learn what the "cost factor" buys you.

## 🛠️ Setup

```bash
cd 01-foundations/authentication-authorization
uv sync
```

Then select the `.venv` kernel in VS Code (top-right of the notebook). If it does not appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 🟥 Approach 1 (BAD): plain text

We just store the password as the user typed it. If the database is ever leaked, every user's password is leaked too — and most people reuse passwords across sites.

In [ ]:
# A toy "users table" — just a Python dict
users_plain = {}

def signup_plain(username, password):
    users_plain[username] = password

def login_plain(username, password):
    return users_plain.get(username) == password

signup_plain("alice", "hunter2")
print("login ok?", login_plain("alice", "hunter2"))
print("login bad?", login_plain("alice", "wrong"))
print("DB contents:", users_plain)  # 😱 the password is right there

# The failure is that the stored value IS the password.
assert users_plain["alice"] == "hunter2"

## 🟨 Approach 2 (BETTER): SHA-256 hash

A **hash function** takes any input and produces a fixed-size fingerprint. It is *one-way*: given the hash you cannot easily get the password back. So we store the hash, not the password.

This is better — but still wrong for passwords. SHA-256 is **fast**, which means an attacker with a leaked database can try **billions of guesses per second** on a GPU. Worse, two users with the same password get the same hash, which makes "rainbow tables" easy.

In [ ]:
import hashlib

def sha256_hex(s: str) -> str:
    return hashlib.sha256(s.encode()).hexdigest()

users_sha = {}

def signup_sha(username, password):
    users_sha[username] = sha256_hex(password)

def login_sha(username, password):
    return users_sha.get(username) == sha256_hex(password)

signup_sha("alice", "hunter2")
signup_sha("bob", "hunter2")  # same password as alice
print("login ok?", login_sha("alice", "hunter2"))
print("DB:", users_sha)
# Notice: alice and bob have IDENTICAL hashes — easy to spot password reuse.

# Same password -> byte-identical stored value. An attacker who cracks one
# account has cracked every account that reused that password.
assert users_sha["alice"] == users_sha["bob"]

## 🕵️ Let's actually crack a SHA-256 hash

Talk is cheap. Let's show *why* fast hashes are bad. We take a leaked SHA-256 hash and try to reverse it by hashing every word in a tiny wordlist. On a real laptop with a real wordlist (or a GPU), this scales to **billions** of guesses per second.

> Note: this is an *illustrative* demo. Real attackers use giant wordlists and optimized GPU code. The point is: SHA-256 is so fast that guessing is cheap.

In [ ]:
import hashlib, time, bcrypt

# Pretend this is what leaked from a database.
leaked_hash = hashlib.sha256(b"sunshine").hexdigest()

# A tiny "dictionary" — real ones have millions of entries.
wordlist = ["password", "123456", "letmein", "qwerty", "sunshine", "dragon", "monkey"]

t0 = time.perf_counter()
cracked = None
for guess in wordlist:
    if hashlib.sha256(guess.encode()).hexdigest() == leaked_hash:
        cracked = guess
        break
sha_ms = (time.perf_counter() - t0) * 1000
assert cracked == "sunshine"
print(f"SHA-256: cracked {leaked_hash[:16]}... -> {cracked!r} in {sha_ms:.3f} ms")

# Now the SAME attack against a bcrypt hash of the SAME password.
# cost=10 here so the notebook stays fast; production is 12+.
leaked_bcrypt = bcrypt.hashpw(b"sunshine", bcrypt.gensalt(rounds=10))
t0 = time.perf_counter()
for guess in wordlist:
    if bcrypt.checkpw(guess.encode(), leaked_bcrypt):
        cracked = guess
        break
bcrypt_ms = (time.perf_counter() - t0) * 1000
print(f"bcrypt : cracked the same password in {bcrypt_ms:.1f} ms")

slowdown = bcrypt_ms / sha_ms
print(f"\nSame 7 guesses, {slowdown:,.0f}x slower.")
print(f"Extrapolated to a 14-million-word list (rockyou.txt):")
print(f"  SHA-256: {14e6 * sha_ms/len(wordlist) / 1000:,.0f} s")
print(f"  bcrypt : {14e6 * bcrypt_ms/len(wordlist) / 3600 / 1000:,.0f} hours "
      f"— per account, because every salt is different")

# The whole security argument is this ratio. Assert it rather than asserting a feeling.
assert slowdown > 100, slowdown

## 🧂 Salt: same password, different stored values

A **salt** is a random value we mix into the password before hashing, and we store the salt alongside the hash. Now two users with the same password produce *different* stored hashes, which defeats pre-computed **rainbow tables**.

Note: salt alone does not make the hash *slow* — SHA-256 is still fast. That's why we still need bcrypt/Argon2 below. Salt and slowness solve *different* problems.

In [ ]:
import os, hashlib

def sha256_salted(password: str, salt: bytes) -> str:
    return hashlib.sha256(salt + password.encode()).hexdigest()

alice_salt = os.urandom(16)
bob_salt   = os.urandom(16)

alice_hash = sha256_salted("hunter2", alice_salt)
bob_hash   = sha256_salted("hunter2", bob_salt)

print("alice:", alice_salt.hex()[:12], "...", alice_hash[:20], "...")
print("bob  :", bob_salt.hex()[:12], "...", bob_hash[:20], "...")
print("same password, DIFFERENT stored hashes ✅")


### 🌶️ Quick aside: "pepper"

A **pepper** is an extra secret value added to the hash input, stored *separately* from the database (e.g. in an environment variable or HSM). If the DB leaks but the pepper doesn't, offline cracking becomes much harder.

Pepper is **optional defense-in-depth**, not a replacement for bcrypt/Argon2. It adds operational complexity (rotation, secret management) and is rarely needed for small projects.

## 🟩 Approach 3 (BEST): bcrypt with salt + cost factor

`bcrypt` was designed for passwords. It does three important things:

1. **Salt** — a random value mixed into the hash so two users with the same password get different hashes.
2. **Slow on purpose** — it loops thousands of times. A "cost factor" of 12 means roughly `2^12 = 4096` rounds.
3. **Tunable** — as hardware gets faster, you bump the cost factor up.

Slow is good here: a legitimate login takes ~100 ms, but an attacker trying billions of guesses now takes years.

In [ ]:
import bcrypt
import time

users_bcrypt = {}

def signup_bcrypt(username, password, cost=12):
    # bcrypt generates a random salt for us and embeds it in the output hash
    hashed = bcrypt.hashpw(password.encode(), bcrypt.gensalt(rounds=cost))
    users_bcrypt[username] = hashed

def login_bcrypt(username, password):
    stored = users_bcrypt.get(username)
    if stored is None:
        return False
    return bcrypt.checkpw(password.encode(), stored)

signup_bcrypt("alice", "hunter2")
signup_bcrypt("bob", "hunter2")  # same password — different hash
for u, h in users_bcrypt.items():
    print(u, "->", h.decode())

In [ ]:
# Time a single login for different cost factors. Higher = slower = more secure.
for cost in (4, 8, 12):
    h = bcrypt.hashpw(b"hunter2", bcrypt.gensalt(rounds=cost))
    t0 = time.perf_counter()
    bcrypt.checkpw(b"hunter2", h)
    dt = (time.perf_counter() - t0) * 1000
    print(f"cost={cost:2d}  verify={dt:7.2f} ms")

## 🚧 The bcrypt footgun: 72 bytes

bcrypt only looks at the **first 72 bytes** of the password. Everything after that is
ignored. This is not a bug in any implementation — it is baked into the algorithm.

It sounds harmless ("who has a 72-character password?") until you build something
like `hash(username + ":" + password)` or `hash(password + pepper)`, or you accept
passphrases, or your users type non-ASCII characters (a single emoji is 4 bytes). Once
the prefix fills 72 bytes, **the password stops mattering** — anyone who knows the
prefix can log in with any suffix.

This is exactly the shape of a real 2024 incident at Okta: a cache key built from
`userId + username + password` meant that for long usernames, the password was never
reached.

Modern `bcrypt` (5.x) refuses rather than truncating. Older versions — and plenty of
bindings in other languages — truncate silently. Let's see both.

In [ ]:
import bcrypt

# 1) Today's library refuses the input outright. Good.
try:
    bcrypt.hashpw(b"A" * 100, bcrypt.gensalt(rounds=6))
    raise AssertionError("expected bcrypt to reject a >72-byte password")
except ValueError as e:
    print("bcrypt 5.x:", e)

# 2) What silent truncation (older bcrypt, other languages) would have done:
prefix = b"verylongusername@example.com:" + b"x" * 45   # 74 bytes before the password
pw_a = prefix + b"correct-horse-battery-staple"
pw_b = prefix + b"hunter2"

stored = bcrypt.hashpw(pw_a[:72], bcrypt.gensalt(rounds=6))   # the old truncation
print(f"\nprefix is {len(prefix)} bytes, so both 'passwords' truncate to the same 72.")
print("does the WRONG password now log in? ->", bcrypt.checkpw(pw_b[:72], stored))

# Two completely different passwords, one accepted hash. Assert the failure.
assert pw_a != pw_b
assert bcrypt.checkpw(pw_b[:72], stored) is True

print("\n✅ Rules: hash the password on its own, never concatenated with anything")
print("   long; if you must combine, HMAC it first (fixed 32 bytes) or use Argon2id,")
print("   which has no length limit.")

## ✅ Recap

| Approach | Salt? | Slow? | Verdict |
|---|---|---|---|
| Plain text | – | – | Never. |
| SHA-256 | No | Very fast | No — easy to crack (see demo above). |
| SHA-256 + salt | Yes | Very fast | Stops rainbow tables, but still too fast. |
| **bcrypt** | **Yes** | **Tunable** | ✅ Great default. |
| **Argon2id** | Yes | Tunable | ✅ Modern winner (PHC 2015). Memory-hard, harder for GPUs/ASICs. |
| **scrypt** | Yes | Tunable | ✅ Also memory-hard. |

**Modern recommendation:** Argon2id if you can, bcrypt if you can't. Use a well-audited library — never write the hashing yourself.

Current OWASP parameters (2024) if you need numbers to put in a config file:

| Algorithm | Parameters | Notes |
|---|---|---|
| **Argon2id** | m=19 MiB, t=2, p=1 (or m=47 MiB, t=1, p=1) | memory-hard; the first choice for new systems |
| **scrypt** | N=2¹⁷, r=8, p=1 | memory-hard; fine if Argon2 isn't available |
| **bcrypt** | work factor **≥ 10** (12 is a common default); password ≤ 72 bytes | see the footgun below |
| **PBKDF2** | ≥ 600,000 iterations (HMAC-SHA-256) | only when FIPS compliance forces it |

Tune upward until a verify takes **~250–500 ms** on *your* hardware, then re-tune every
couple of years. Store the parameters with the hash (all of these formats do) so you
can re-hash on next login when you raise them.

**When comparing hashes, use a constant-time compare** to avoid timing side-channels:

- `bcrypt.checkpw` already does this internally.
- If you ever compare raw hex/bytes by hand, use `hmac.compare_digest(a, b)`, **not** `a == b`.

### 📚 Real-world context (why this matters)

- **2012 LinkedIn breach:** ~6.5M passwords leaked as **unsalted SHA-1**. Most were cracked within days.
- **2013 Adobe breach:** ~150M passwords stored with a broken scheme and reversible encryption; password *hints* were stored in plain text.
- **Rule of thumb:** assume *every* password DB will eventually leak. Your job is to make the leak useless.
